In [1]:
import pickle

import numpy as np
import pandas as pd
import torch
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
class CreditRiskMLP(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_layer_sizes=(128, 64),
        use_batch_norm=False,
        activation='relu',
    ):
        super().__init__()

        activation_layers = {
            'relu': nn.ReLU,
            'leaky_relu': nn.LeakyReLU,
            'tanh': nn.Tanh,
        }

        if activation not in activation_layers:
            raise ValueError(f'Unknown activation: {activation}')

        activation_layer = activation_layers[activation]

        layers = []
        previous_size = input_size

        for hidden_size in hidden_layer_sizes:
            layers.append(nn.Linear(previous_size, hidden_size))

            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_size))

            layers.append(activation_layer())
            previous_size = hidden_size

        layers.append(nn.Linear(previous_size, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

In [3]:
class TorchMLPClassifier(ClassifierMixin, BaseEstimator):
    def __init__(
        self,
        hidden_layer_sizes=(128, 64),
        batch_size=4096,
        max_epochs=10,
        learning_rate=0.001,
        weight_decay=0.0,
        use_pos_weight=False,
        early_stopping=False,
        use_batch_norm=False,
        validation_fraction=0.1,
        patience=2,
        random_state=42,
        device='cpu',
        verbose=True,
        activation='relu',
    ):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.use_pos_weight = use_pos_weight
        self.early_stopping = early_stopping
        self.use_batch_norm = use_batch_norm
        self.validation_fraction = validation_fraction
        self.patience = patience
        self.random_state = random_state
        self.device = device
        self.verbose = verbose
        self.activation = activation


    def fit(self, X, y):
        self.classes_ = np.array([0, 1])  # для нормальной sklearn-совместимости
        X, y = np.asarray(X), np.asarray(y)

        X_train, X_val, y_train, y_val = train_test_split(
            X, y,
            test_size=self.validation_fraction,
            random_state=self.random_state,
            stratify=y,
        )

        X_train_tensor = torch.from_numpy(X_train).float()
        y_train_tensor = torch.from_numpy(y_train).float()
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

        X_val_tensor = torch.from_numpy(X_val).float()
        y_val_tensor = torch.from_numpy(y_val).float()
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

        train_generator = torch.Generator()
        train_generator.manual_seed(self.random_state)
        pin_memory = self.device == 'cuda'

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            pin_memory=pin_memory,
            generator=train_generator
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            pin_memory=pin_memory,
        )

        torch.manual_seed(self.random_state)

        self.model_ = CreditRiskMLP(
            input_size=X.shape[1],
            hidden_layer_sizes=self.hidden_layer_sizes,
            use_batch_norm=self.use_batch_norm,
            activation=self.activation,
        ).to(self.device)

        if self.use_pos_weight:
            n_positive = y_train_tensor.sum().item()
            n_negative = len(y_train_tensor) - n_positive

            pos_weight = torch.tensor(
                n_negative / n_positive,
                dtype=torch.float32,
                device=self.device
            )
        else:
            pos_weight = None

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay
        )

        epoch_iterator = (
            tqdm(range(self.max_epochs), desc='Training', unit='epoch', ncols=100)
            if self.verbose
            else range(self.max_epochs)
        )

        self.val_auc_history_ = []
        if self.early_stopping:
            best_auc = -np.inf
            best_state = None
            epochs_without_improvement = 0

        for epoch in epoch_iterator:
            self.model_.train()
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device, non_blocking=pin_memory)
                y_batch = y_batch.to(self.device, non_blocking=pin_memory)
                optimizer.zero_grad()
                pred_batch = self.model_(X_batch)
                loss_batch = criterion(pred_batch, y_batch)
                loss_batch.backward()
                optimizer.step()

            val_auc = self._calc_auc(val_loader)
            self.val_auc_history_.append(val_auc)

            if self.early_stopping:
                if val_auc > best_auc:
                    best_auc = val_auc
                    best_state = copy.deepcopy(self.model_.state_dict())
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1

                if epochs_without_improvement >= self.patience:
                    break

        if self.early_stopping:
            self.model_.load_state_dict(best_state)
        return self


    def _calc_auc(self, val_loader):
        self.model_.eval()
        y_true, y_pred = [], []
        pin_memory = self.device == 'cuda'

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(self.device, non_blocking=pin_memory)
                pred_batch = self.model_(X_batch)

                y_true.append(y_batch.numpy())
                y_pred.append(pred_batch.cpu().numpy())

        return roc_auc_score(
            np.concatenate(y_true),
            np.concatenate(y_pred)
        )


    def predict_proba(self, X):
        check_is_fitted(self, 'model_')
        self.model_.eval()
        X_tensor = torch.from_numpy(np.asarray(X, dtype=np.float32))
        pin_memory = self.device == 'cuda'
        loader = DataLoader(
            TensorDataset(X_tensor),
            batch_size=self.batch_size,
            shuffle=False,
            pin_memory=pin_memory
        )
        proba = []
        with torch.no_grad():
            for (X_batch,) in loader:
                X_batch = X_batch.to(self.device, non_blocking=pin_memory)
                logits_batch = self.model_(X_batch)
                proba_batch = torch.sigmoid(logits_batch)
                proba.append(proba_batch.cpu().numpy())

        proba = np.concatenate(proba)
        return np.column_stack((1 - proba, proba))


    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)

In [4]:
def engineer_features(X, categorical_features):
    X = X.copy()

    X['log1p(credit_count)'] = (
        np.log1p(X['credit_count']).astype('float32')
    )
    X = X.drop(columns='credit_count')

    for feature in categorical_features:
        cols = [
            col for col in X.columns
            if col.startswith(f'{feature}_')
        ]
        X[f'{feature}_nunique'] = (
            (X[cols] > 0).sum(axis=1).astype('uint8')
        )

    return X

In [5]:
# Проверяем загрузку пайплайна из файла

# Поскольку X_check и predict_proba_check сохранены в файлы,
# проверку загрузки пайплайна можно выполнить после перезапуска ядра.
# Перед проверкой нужно выполнить только импорты и определения
# CreditRiskMLP, TorchMLPClassifier и engineer_features.

with open('credit_risk_pipeline.pkl', 'rb') as file:
    loaded_pipeline = pickle.load(file)

X_check = pd.read_parquet('X_check.pq')
predict_proba_check = np.load('predict_proba_check.npy')
loaded_predict_proba = loaded_pipeline.predict_proba(X_check)[:, 1]

predictions_match = np.allclose(
    predict_proba_check,
    loaded_predict_proba
)
print(
    'Загруженный пайплайн даёт такие же предсказания: '
    f'{predictions_match}'
)

Загруженный пайплайн даёт такие же предсказания: True
